# Pydantic + LangChain Tools with Gemini API

이 노트북은 Google Gemini API와 Pydantic 모델을 사용하여 구조화된 입력을 받는 LangChain Tools를 보여줍니다.

## 주요 학습 내용:
- Pydantic 모델을 사용한 구조화된 입력 정의
- 타입 안전성과 데이터 검증
- 복합 도구 시스템 (시간 조회 + 주식 조회)
- Mock 데이터를 활용한 실제 API 시뮬레이션

## 시나리오:
사용자가 주식 정보를 요청하면 AI가 Pydantic으로 검증된 데이터로 주식 조회 도구를 사용하여 답변

## 1. 환경 설정 및 모델 초기화

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from datetime import datetime
import pytz
from pydantic import BaseModel, Field

In [ ]:
# 환경변수 로드
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("환경변수 GOOGLE_API_KEY가 설정되지 않았습니다.")

print("✅ 환경변수가 성공적으로 로드되었습니다.")

✅ 환경변수가 성공적으로 로드되었습니다.


In [3]:
# Gemini 모델 초기화
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.7,
)

print("✅ Gemini 모델이 초기화되었습니다.")

✅ Gemini 모델이 초기화되었습니다.


## 2. 기본 채팅 테스트

In [4]:
# 기본 채팅 테스트
response = llm.invoke([HumanMessage("잘 지냈어?")])
print(f"🤖 Gemini 응답: {response.content}")

🤖 Gemini 응답: 네, 잘 지냈어요! 덕분에 오늘도 활기차게 여러 질문에 답변하고 있습니다. 😊 혹시 오늘 특별히 궁금한 점이나 필요한 정보가 있으신가요?


## 3. 기본 도구: 시간 조회

In [5]:
@tool  # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    try:
        tz = pytz.timezone(timezone)
        now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
        location_and_local_time = f'{timezone} ({location}) 현재시각 {now}'
        print(f"🕐 시간 조회: {location_and_local_time}")
        return location_and_local_time
    except Exception as e:
        error_msg = f"시간 조회 실패: {str(e)}"
        print(f"❌ {error_msg}")
        return error_msg

print("✅ 시간 조회 도구가 정의되었습니다.")

✅ 시간 조회 도구가 정의되었습니다.


## 4. Pydantic 모델 정의

In [6]:
class StockHistoryInput(BaseModel):
    """주식 조회를 위한 입력 모델"""
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL, TSLA, MSFT)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 5d, 1mo, 3mo, 6mo, 1y)")

print("✅ StockHistoryInput 모델이 정의되었습니다.")
print(f"📋 필수 필드:")
for field_name, field in StockHistoryInput.model_fields.items():
    title = getattr(field, 'title', field_name)
    description = getattr(field, 'description', '설명 없음')
    print(f"  - {field_name}: {title} - {description}")

✅ StockHistoryInput 모델이 정의되었습니다.
📋 필수 필드:
  - ticker: 주식 코드 - 주식 코드 (예: AAPL, TSLA, MSFT)
  - period: 기간 - 주식 데이터 조회 기간 (예: 1d, 5d, 1mo, 3mo, 6mo, 1y)


## 5. 주식 조회 도구 (실제 yfinance 사용)

In [7]:
@tool
def get_stock_history(stock_history_input: StockHistoryInput) -> str:
    """주식 종목의 가격 데이터를 조회하는 함수 (실제 yfinance 사용)
    
    yfinance를 사용하여 실시간 주식 데이터를 가져옵니다.
    """
    try:
        import yfinance as yf
        
        ticker = stock_history_input.ticker.upper()
        period = stock_history_input.period
        
        print(f"📈 주식 조회: {ticker} ({period})")
        
        # yfinance로 주식 데이터 가져오기
        stock = yf.Ticker(ticker)
        hist = stock.history(period=period)
        
        if hist.empty:
            return f"❌ {ticker} 주식 데이터를 찾을 수 없습니다. 티커를 확인해주세요."
        
        # 최근 5개 데이터 포맷팅
        recent_data = hist.tail(5)
        
        # 결과 포맷팅
        result = f"📊 {ticker} 주식 데이터 ({period}):\n\n"
        result += "| Date       | Open   | High   | Low    | Close  | Volume     |\n"
        result += "|------------|--------|--------|--------|--------|------------|\n"
        
        for date, row in recent_data.iterrows():
            date_str = date.strftime('%Y-%m-%d')
            open_price = f"${row['Open']:.2f}"
            high_price = f"${row['High']:.2f}"
            low_price = f"${row['Low']:.2f}"
            close_price = f"${row['Close']:.2f}"
            volume = f"{row['Volume']:,}"
            
            result += f"| {date_str} | {open_price:>7} | {high_price:>7} | {low_price:>7} | {close_price:>7} | {volume:>10} |\n"
        
        # 현재 가격 정보 추가
        current_price = stock.info.get('currentPrice', hist['Close'].iloc[-1])
        result += f"\n💰 현재 가격: ${current_price:.2f}"
        
        return result
        
    except ImportError:
        return "❌ yfinance 패키지가 설치되지 않았습니다. 'pip install yfinance'로 설치해주세요."
    except Exception as e:
        error_msg = f"주식 데이터 조회 실패: {str(e)}"
        print(f"❌ {error_msg}")
        return error_msg

print("✅ 주식 조회 도구가 정의되었습니다.")

✅ 주식 조회 도구가 정의되었습니다.


## 6. 도구를 모델에 바인딩

In [8]:
# 도구 리스트 및 딕셔너리 구성
tools = [get_current_time, get_stock_history]
tool_dict = {
    "get_current_time": get_current_time,
    "get_stock_history": get_stock_history
}

# 도구를 모델에 바인딩
llm_with_tools = llm.bind_tools(tools)

print("✅ 도구들이 모델에 바인딩되었습니다.")
print(f"📋 사용 가능한 도구: {[tool.name for tool in tools]}")

✅ 도구들이 모델에 바인딩되었습니다.
📋 사용 가능한 도구: ['get_current_time', 'get_stock_history']


## 7. 시간 조회 테스트

In [9]:
# 부산 시간 조회 테스트
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

print("💬 사용자 질문: 부산은 지금 몇시야?")

response = llm_with_tools.invoke(messages)
messages.append(response)

print(f"\n🤖 AI 응답: {response.content}")

if hasattr(response, 'tool_calls') and response.tool_calls:
    print(f"🔧 호출된 도구 수: {len(response.tool_calls)}")
    
    for tool_call in response.tool_calls:
        selected_tool = tool_dict[tool_call["name"]]
        print(f"🛠️ 도구 호출: {tool_call['name']}")
        print(f"📥 전달된 인자: {tool_call['args']}")
        
        tool_msg = selected_tool.invoke(tool_call)
        messages.append(tool_msg)
    
    final_response = llm_with_tools.invoke(messages)
    print(f"🎯 최종 답변: {final_response.content}")
else:
    print("ℹ️ 도구 호출이 없었습니다.")

💬 사용자 질문: 부산은 지금 몇시야?

🤖 AI 응답: 
🔧 호출된 도구 수: 1
🛠️ 도구 호출: get_current_time
📥 전달된 인자: {'location': '부산', 'timezone': 'Asia/Seoul'}
🕐 시간 조회: Asia/Seoul (부산) 현재시각 2025-09-04 10:54:04
🎯 최종 답변: 부산은 지금 2025년 9월 4일 10시 54분입니다.


## 8. 주식 조회 테스트 - 테슬라

In [10]:
# 새로운 대화 시작
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"),
]

print("💬 사용자 질문: 테슬라는 한달 전에 비해 주가가 올랐나 내렸나?")

response = llm_with_tools.invoke(messages)
messages.append(response)

print(f"\n🤖 AI 응답: {response.content}")

if hasattr(response, 'tool_calls') and response.tool_calls:
    print(f"\n🔧 호출된 도구 수: {len(response.tool_calls)}")
    
    for tool_call in response.tool_calls:
        selected_tool = tool_dict[tool_call["name"]]
        print(f"🛠️ 도구 호출: {tool_call['name']}")
        print(f"📥 전달된 인자: {tool_call['args']}")
        
        tool_msg = selected_tool.invoke(tool_call)
        messages.append(tool_msg)
        print(f"📤 도구 실행 결과 (일부): {tool_msg.content[:200]}...")
else:
    print("ℹ️ 도구 호출이 없었습니다.")

print(f"\n📋 현재 메시지 수: {len(messages)}")

💬 사용자 질문: 테슬라는 한달 전에 비해 주가가 올랐나 내렸나?

🤖 AI 응답: 

🔧 호출된 도구 수: 1
🛠️ 도구 호출: get_stock_history
📥 전달된 인자: {'stock_history_input': {'period': '1mo', 'ticker': 'TSLA'}}
📈 주식 조회: TSLA (1mo)
📤 도구 실행 결과 (일부): 📊 TSLA 주식 데이터 (1mo):

| Date       | Open   | High   | Low    | Close  | Volume     |
|------------|--------|--------|--------|--------|------------|
| 2025-08-27 | $351.94 | $355.39 | $349.16 | $349....

📋 현재 메시지 수: 4


In [11]:
# 도구 실행 결과를 바탕으로 최종 답변 생성
if len(messages) > 2:  # 도구 실행 결과가 있다면
    print("🔄 도구 실행 결과를 바탕으로 최종 답변 생성 중...")
    final_response = llm_with_tools.invoke(messages)
    print(f"\n🎯 최종 답변: {final_response.content}")
else:
    print("💬 도구 실행 없이 직접 답변했습니다.")

🔄 도구 실행 결과를 바탕으로 최종 답변 생성 중...

🎯 최종 답변: 한 달 전 테슬라 주가는 349.60달러였는데 현재 334.09달러입니다. 따라서 한 달 전에 비해 주가가 내렸습니다.


## 9. 여러 주식 조회 테스트

In [12]:
# 여러 주식에 대한 질문들
stock_questions = [
    "애플 주식의 최근 1개월 성과는?",
    "마이크로소프트 1년 주가 변동은?",
    "구글 주식 정보를 알려줘",  # Mock 데이터 없는 경우 테스트
]

print("📈 여러 주식 조회 테스트")
print("=" * 40)

for question in stock_questions:
    print(f"\n💬 질문: {question}")
    
    messages = [
        SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
        HumanMessage(question),
    ]
    
    try:
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        if hasattr(response, 'tool_calls') and response.tool_calls:
            for tool_call in response.tool_calls:
                selected_tool = tool_dict[tool_call["name"]]
                tool_msg = selected_tool.invoke(tool_call)
                messages.append(tool_msg)
            
            final_response = llm_with_tools.invoke(messages)
            print(f"🎯 답변: {final_response.content[:300]}...")
        else:
            print(f"🤖 답변: {response.content}")
            
    except Exception as e:
        print(f"❌ 오류 발생: {str(e)}")

📈 여러 주식 조회 테스트

💬 질문: 애플 주식의 최근 1개월 성과는?
📈 주식 조회: AAPL (1mo)
🎯 답변: 최근 1개월 동안 애플 주식은 228.61달러로 시작하여 238.47달러로 마감했습니다....

💬 질문: 마이크로소프트 1년 주가 변동은?
📈 주식 조회: MSFT (1y)
🎯 답변: 마이크로소프트의 1년간 주가 변동은 다음과 같습니다. 2025년 8월 27일에 $506.74로 시작하여 2025년 9월 3일에 $505.35로 마감했습니다....

💬 질문: 구글 주식 정보를 알려줘
🤖 답변: 어떤 기간으로 조회할까요? (예: 1d, 5d, 1mo, 3mo, 6mo, 1y)


## 10. Pydantic 모델 검증 테스트

In [13]:
# 올바른 데이터로 모델 생성 테스트
print("✅ 올바른 데이터 테스트:")
try:
    valid_input = StockHistoryInput(ticker="TSLA", period="1mo")
    print(f"  - 생성 성공: {valid_input}")
    print(f"  - 주식 코드: {valid_input.ticker}")
    print(f"  - 기간: {valid_input.period}")
    print(f"  - 모델 타입: {type(valid_input)}")
except Exception as e:
    print(f"  - 생성 실패: {e}")

✅ 올바른 데이터 테스트:
  - 생성 성공: ticker='TSLA' period='1mo'
  - 주식 코드: TSLA
  - 기간: 1mo
  - 모델 타입: <class '__main__.StockHistoryInput'>


In [14]:
# 잘못된 데이터로 모델 생성 테스트
print("❌ 잘못된 데이터 테스트:")
try:
    # 필수 필드 누락
    invalid_input = StockHistoryInput(ticker="AAPL")  # period 누락
    print(f"  - 생성 성공 (예상치 못함): {invalid_input}")
except Exception as e:
    print(f"  - 생성 실패 (예상됨): {e}")
    print("  - ✅ Pydantic이 올바르게 검증했습니다!")

❌ 잘못된 데이터 테스트:
  - 생성 실패 (예상됨): 1 validation error for StockHistoryInput
period
  Field required [type=missing, input_value={'ticker': 'AAPL'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
  - ✅ Pydantic이 올바르게 검증했습니다!


In [15]:
# 다양한 올바른 데이터 테스트
print("🧪 다양한 올바른 데이터 테스트:")

test_cases = [
    {"ticker": "AAPL", "period": "1d"},
    {"ticker": "msft", "period": "1y"},  # 소문자도 처리되는지 확인
    {"ticker": "GOOGL", "period": "6mo"},
]

for i, case in enumerate(test_cases, 1):
    try:
        model = StockHistoryInput(**case)
        print(f"  {i}. ✅ {model.ticker} ({model.period}) - 성공")
    except Exception as e:
        print(f"  {i}. ❌ {case} - 실패: {e}")

🧪 다양한 올바른 데이터 테스트:
  1. ✅ AAPL (1d) - 성공
  2. ✅ msft (1y) - 성공
  3. ✅ GOOGL (6mo) - 성공


## 11. 직접 도구 호출 테스트

In [16]:
# 주식 조회 도구를 직접 호출해보기
print("🔧 주식 조회 도구 직접 호출 테스트")
print("=" * 40)

test_stocks = [
    StockHistoryInput(ticker="TSLA", period="1mo"),
    StockHistoryInput(ticker="AAPL", period="1y"),
    StockHistoryInput(ticker="MSFT", period="1mo"),
    StockHistoryInput(ticker="GOOGL", period="1mo"),  # 실제 yfinance 사용
]

for stock_input in test_stocks:
    print(f"\n📊 {stock_input.ticker} ({stock_input.period}) 조회:")
    result = get_stock_history.invoke({"stock_history_input": stock_input})
    print(f"📈 결과 (일부): {result[:200]}...")

🔧 주식 조회 도구 직접 호출 테스트

📊 TSLA (1mo) 조회:
📈 주식 조회: TSLA (1mo)
📈 결과 (일부): 📊 TSLA 주식 데이터 (1mo):

| Date       | Open   | High   | Low    | Close  | Volume     |
|------------|--------|--------|--------|--------|------------|
| 2025-08-27 | $351.94 | $355.39 | $349.16 | $349....

📊 AAPL (1y) 조회:
📈 주식 조회: AAPL (1y)
📈 결과 (일부): 📊 AAPL 주식 데이터 (1y):

| Date       | Open   | High   | Low    | Close  | Volume     |
|------------|--------|--------|--------|--------|------------|
| 2025-08-27 | $228.61 | $230.90 | $228.26 | $230.4...

📊 MSFT (1mo) 조회:
📈 주식 조회: MSFT (1mo)
📈 결과 (일부): 📊 MSFT 주식 데이터 (1mo):

| Date       | Open   | High   | Low    | Close  | Volume     |
|------------|--------|--------|--------|--------|------------|
| 2025-08-27 | $502.00 | $507.29 | $499.90 | $506....

📊 GOOGL (1mo) 조회:
📈 주식 조회: GOOGL (1mo)
📈 결과 (일부): 📊 GOOGL 주식 데이터 (1mo):

| Date       | Open   | High   | Low    | Close  | Volume     |
|------------|--------|--------|--------|--------|------------|
| 2025-08-27 | $205.

## 12. 사용법 및 확장 정보

In [17]:
print("📈 실제 yfinance를 사용한 주식 데이터:")
print("  - 실시간 주식 가격 데이터 제공")
print("  - 모든 주요 거래소 지원 (NASDAQ, NYSE 등)")
print("  - 다양한 주식 코드 지원 (TSLA, AAPL, MSFT, GOOGL 등)")

print("\n📅 지원하는 기간:")
periods = ["1d: 1일", "5d: 5일", "1mo: 1개월", "3mo: 3개월", "6mo: 6개월", "1y: 1년"]
for period in periods:
    print(f"  - {period}")

print("\n🔧 사용 전 설정:")
steps = [
    "1. yfinance 패키지 설치: pip install yfinance",
    "2. 인터넷 연결 확인 (실시간 데이터 조회)",
    "3. 올바른 주식 티커 심볼 사용 확인"
]
for step in steps:
    print(f"  {step}")

print("\n⚠️ 참고사항:")
notes = [
    "- yfinance 미설치 시 자동으로 에러 메시지 표시",
    "- 잘못된 티커 심볼의 경우 데이터 없음 메시지", 
    "- 시장 휴장일에는 최신 거래일 데이터 반환"
]
for note in notes:
    print(f"  {note}")

📈 실제 yfinance를 사용한 주식 데이터:
  - 실시간 주식 가격 데이터 제공
  - 모든 주요 거래소 지원 (NASDAQ, NYSE 등)
  - 다양한 주식 코드 지원 (TSLA, AAPL, MSFT, GOOGL 등)

📅 지원하는 기간:
  - 1d: 1일
  - 5d: 5일
  - 1mo: 1개월
  - 3mo: 3개월
  - 6mo: 6개월
  - 1y: 1년

🔧 사용 전 설정:
  1. yfinance 패키지 설치: pip install yfinance
  2. 인터넷 연결 확인 (실시간 데이터 조회)
  3. 올바른 주식 티커 심볼 사용 확인

⚠️ 참고사항:
  - yfinance 미설치 시 자동으로 에러 메시지 표시
  - 잘못된 티커 심볼의 경우 데이터 없음 메시지
  - 시장 휴장일에는 최신 거래일 데이터 반환


## 13. yfinance 패키지 설치 및 사용법

In [18]:
# yfinance 패키지 설치 가이드
print("💡 yfinance 패키지 설치 및 사용:")
print("")

install_guide = '''
# 1. yfinance 설치 (터미널에서 실행)
pip install yfinance

# 또는 conda 사용시
conda install -c conda-forge yfinance

# 2. 현재 노트북에서 yfinance가 작동하는지 확인
import yfinance as yf
stock = yf.Ticker("AAPL")
info = stock.info
print(f"애플 현재가: ${info.get('currentPrice', '데이터 없음')}")
'''

print(install_guide)
print("\n📊 yfinance로 가능한 작업들:")
features = [
    "- 실시간 주식 가격 조회",
    "- 과거 주가 데이터 (일별, 주별, 월별)",
    "- 기업 정보 (시가총액, PER, 배당률 등)",
    "- 재무제표 데이터",
    "- 옵션 데이터",
    "- 뉴스 및 이벤트 정보"
]
for feature in features:
    print(f"  {feature}")

print("\n⚠️ 주의사항:")
warnings = [
    "- 인터넷 연결이 필요합니다",
    "- Yahoo Finance의 데이터 정책에 따라 제한될 수 있습니다",
    "- 실시간 데이터는 약간의 지연이 있을 수 있습니다"
]
for warning in warnings:
    print(f"  {warning}")

💡 yfinance 패키지 설치 및 사용:


# 1. yfinance 설치 (터미널에서 실행)
pip install yfinance

# 또는 conda 사용시
conda install -c conda-forge yfinance

# 2. 현재 노트북에서 yfinance가 작동하는지 확인
import yfinance as yf
stock = yf.Ticker("AAPL")
info = stock.info
print(f"애플 현재가: ${info.get('currentPrice', '데이터 없음')}")


📊 yfinance로 가능한 작업들:
  - 실시간 주식 가격 조회
  - 과거 주가 데이터 (일별, 주별, 월별)
  - 기업 정보 (시가총액, PER, 배당률 등)
  - 재무제표 데이터
  - 옵션 데이터
  - 뉴스 및 이벤트 정보

⚠️ 주의사항:
  - 인터넷 연결이 필요합니다
  - Yahoo Finance의 데이터 정책에 따라 제한될 수 있습니다
  - 실시간 데이터는 약간의 지연이 있을 수 있습니다


## 14. 요약 및 학습 정리

In [19]:
print("🎉 Pydantic + LangChain Tools with Gemini API 완료!")
print("=" * 60)

print("\n📚 학습한 주요 개념:")
concepts = [
    "1️⃣ Pydantic BaseModel을 사용한 구조화된 입력 정의",
    "2️⃣ Field를 사용한 상세한 필드 정보와 검증", 
    "3️⃣ 타입 안전성과 자동 데이터 검증",
    "4️⃣ 복합 도구 시스템 (시간 조회 + 주식 조회)",
    "5️⃣ yfinance를 활용한 실시간 주식 데이터 조회"
]
for concept in concepts:
    print(f"  {concept}")

print("\n🔧 구현한 주요 기능:")
features = [
    "- Pydantic 모델을 통한 입력 검증",
    "- 실시간 주식 데이터 조회 (yfinance 사용)",
    "- 전 세계 시간대별 현재 시간 조회", 
    "- 다중 도구 자동 선택 및 실행",
    "- 에러 처리 및 복구 메커니즘 (yfinance 미설치 대응)"
]
for feature in features:
    print(f"  {feature}")

print("\n🛠️ 사용한 주요 기술:")
technologies = [
    "- Google Gemini 2.0 Flash 모델",
    "- Pydantic v2 모델과 필드 검증",
    "- LangChain Tools 프레임워크",
    "- yfinance 라이브러리 (실시간 주식 데이터)",
    "- pytz 타임존 라이브러리"
]
for tech in technologies:
    print(f"  {tech}")

print("\n💡 핵심 포인트:")
points = [
    "✅ Pydantic으로 타입 안전성과 데이터 검증 보장",
    "✅ 구조화된 입력으로 도구 호출의 정확성 향상",
    "✅ AI가 필요에 따라 적절한 도구를 자동 선택",
    "✅ 실제 yfinance API로 실시간 주식 데이터 조회 가능",
    "✅ yfinance 미설치 시 친화적인 에러 메시지 제공"
]
for point in points:
    print(f"  {point}")

print("\n🚀 확장 가능한 방향:")
extensions = [
    "- 다양한 금융 지표 추가 (PER, PEG, 배당률 등)",
    "- 포트폴리오 성과 분석 기능",
    "- 기술적 분석 지표 계산 (이동평균, RSI, MACD)",
    "- 다중 주식 비교 분석",
    "- 알림 및 투자 추천 시스템"
]
for ext in extensions:
    print(f"  {ext}")

🎉 Pydantic + LangChain Tools with Gemini API 완료!

📚 학습한 주요 개념:
  1️⃣ Pydantic BaseModel을 사용한 구조화된 입력 정의
  2️⃣ Field를 사용한 상세한 필드 정보와 검증
  3️⃣ 타입 안전성과 자동 데이터 검증
  4️⃣ 복합 도구 시스템 (시간 조회 + 주식 조회)
  5️⃣ yfinance를 활용한 실시간 주식 데이터 조회

🔧 구현한 주요 기능:
  - Pydantic 모델을 통한 입력 검증
  - 실시간 주식 데이터 조회 (yfinance 사용)
  - 전 세계 시간대별 현재 시간 조회
  - 다중 도구 자동 선택 및 실행
  - 에러 처리 및 복구 메커니즘 (yfinance 미설치 대응)

🛠️ 사용한 주요 기술:
  - Google Gemini 2.0 Flash 모델
  - Pydantic v2 모델과 필드 검증
  - LangChain Tools 프레임워크
  - yfinance 라이브러리 (실시간 주식 데이터)
  - pytz 타임존 라이브러리

💡 핵심 포인트:
  ✅ Pydantic으로 타입 안전성과 데이터 검증 보장
  ✅ 구조화된 입력으로 도구 호출의 정확성 향상
  ✅ AI가 필요에 따라 적절한 도구를 자동 선택
  ✅ 실제 yfinance API로 실시간 주식 데이터 조회 가능
  ✅ yfinance 미설치 시 친화적인 에러 메시지 제공

🚀 확장 가능한 방향:
  - 다양한 금융 지표 추가 (PER, PEG, 배당률 등)
  - 포트폴리오 성과 분석 기능
  - 기술적 분석 지표 계산 (이동평균, RSI, MACD)
  - 다중 주식 비교 분석
  - 알림 및 투자 추천 시스템
